# ITCC508 Lab Exercise PT-M1
## Building and Evaluating a Domain-Specific RAG Chatbot

**Course:** ITC CS08 — Introduction to LLMOps and RAG Concepts
**Topic:** Retrieval-Augmented Generation (RAG) & Hallucination Mitigation
**Chosen Domain:** Gaming Lore — *Arcane* (League of Legends universe)

**Problem Statement:** General-purpose LLMs are frequently trained on large amounts of internet fan-content about popular franchises, which mixes canon lore with fan theories, misremembered details, and outright fabrications. This makes it hard to trust an LLM's answers about a specific franchise's "true" story details. This chatbot solves that problem by grounding the LLM strictly in a curated set of accurate *Arcane* lore documents, so that answers are traceable to a specific, verifiable source rather than to whatever the model happened to memorize during pre-training.

**Source Files (stored in a `my_data` folder in Google Drive, at `/content/drive/MyDrive/my_data`):**
1. `arcane_character_profiles.txt` — biographies of the main characters (Vi, Jinx/Powder, Silco, Jayce, Viktor, Caitlyn).
2. `hextech_technology.txt` — the in-universe technology of Hextech: its origins, the Hex Crystal, notable inventions, and the Hexcore.
3. `piltover_zaun_setting.txt` — the world-building: the twin cities of Piltover and Zaun, their conflict, and the drug Shimmer.


### Cell 1: Environment Setup & Package Installation

This cell runs a shell command from inside the notebook the `!` prefix tells Jupyter/Colab to hand the line to the underlying OS shell instead of the Python interpreter. It installs every third-party library the rest of the notebook depends on. Colab and Jupyter runtimes start out bare, so none of these packages exist yet; if this cell is skipped, every `import` statement further down will fail with a `ModuleNotFoundError`. The `-q` flag just keeps the install logs quiet so the notebook output stays readable for grading.

Here's what each package is for and why it's needed:

- **langchain** — the core orchestration framework. It provides the base abstractions (documents, chains, prompts) that every other LangChain package builds on, so nothing downstream would work without it.
- **langchain-classic** — a newer, separate package that LangChain split the older `create_retrieval_chain` and `create_stuff_documents_chain` helpers into. Recent LangChain versions moved this "classic" chain-building functionality out of the core `langchain` package, so it now has to be installed and imported on its own this is what Cell 6 relies on.
- **langchain-community** — LangChain's community-maintained integrations package. It supplies `DirectoryLoader` and `Chroma`, used further down, because these wrap third-party tools rather than being part of core LangChain.
- **langchain-groq** — LangChain's official Groq integration. It supplies the `ChatGroq` class used later to call our chat model on Groq's hosted inference API. Groq is used here because it offers very low-latency inference for open-weight models.
- **langchain-huggingface** — LangChain's Hugging Face integration. It supplies `HuggingFaceEmbeddings`, used to turn text into vectors, letting us use a free, open-source embedding model instead of a paid embedding API.
- **chromadb** — an open-source vector database. It stores the embedded document chunks and lets us search them by similarity this is the "retrieval" half of Retrieval-Augmented Generation, and without it there is nothing to retrieve from.
- **pypdf** — a PDF-parsing library, included so document loaders can extract text from `.pdf` files if any end up in the data folder. It keeps the pipeline general-purpose even though our chosen domain files are `.txt`.
- **unstructured** — a general-purpose document parsing library used internally by `DirectoryLoader` to read and normalize different file types (including plain `.txt`) into LangChain `Document` objects. It's what actually powers the "load anything from a folder" behavior relied on when we load our documents from Google Drive.


In [5]:
# Cell 1: Environment Setup & Package Installation
!pip install -q langchain langchain-classic langchain-community langchain-groq langchain-huggingface chromadb pypdf unstructured

### Explanation — Cell 2: Secure API Key Configuration

`import os` brings in Python's built-in operating-system interface module, which lets the notebook read and write environment variables through `os.environ`. This matters because LangChain's `ChatGroq` class automatically looks for the Groq API key inside the environment variable `GROQ_API_KEY`, so `os` is what lets us set that value.

`import getpass` brings in Python's built-in secure-input module. It provides `getpass.getpass()`, a function that reads input from the user **without echoing it to the screen**, which keeps the API key from showing up in the notebook's visible output or cell history a basic but important credential-security practice.

The line `if "GROQ_API_KEY" not in os.environ:` is a conditional check that only prompts for the key if it hasn't already been set in this session. This avoids re-prompting every time the cell is accidentally re-run, and still allows the key to be supplied another way if preferred.

Finally, `os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")` captures the typed key and stores it as an environment variable for the rest of the session. This is the single place the key is ever handled since it's never printed or hard-coded into a cell, it won't accidentally leak if the notebook file itself is shared or committed to a repository.


In [7]:
# Cell 2: Secure API Key Configuration
import os
import getpass
# Prompts for API key securely without saving or echoing plain text in notebook cells
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

### Explanation — Cell 3: Google Drive Mount, Data Loading & Chunking

This notebook reads the *Arcane* lore files straight from Google Drive instead of writing them into the runtime from Python strings. This is more convenient to re-run: as long as the `my_data` folder already exists in Drive, you never have to re-upload or re-create the source files every time the runtime resets — you only need to mount your Drive and point the loader at the right folder.

`from google.colab import drive` imports Colab's Drive-mounting helper. This is a Colab-specific module, which is why this whole cell only works when the notebook is actually running on Colab rather than a plain local Jupyter server.

`drive.mount('/content/drive')` mounts my Google Drive into the Colab virtual machine's filesystem at `/content/drive`, and will prompt me to authorize access the first time it runs in a session. Once mounted, any file in your Drive becomes readable through a normal file path, which is what lets `DirectoryLoader` treat it exactly like a local folder.

`data_path = "/content/drive/MyDrive/my_data"` defines the path to the folder inside my Drive where the three *Arcane* lore `.txt` files live. This must exactly match the folder's real location and name in my Drive, otherwise the loader below will simply find zero files and every later step embedding, retrieval, generation will have nothing to work with. This is the single most common reason a Drive-based version of this notebook fails to run.

`loader = DirectoryLoader(data_path, glob="**/*.txt", show_progress=True)` configures the loader to recursively scan `data_path` and pick up only `.txt` files. The `glob="**/*.txt"` pattern is narrower than a plain `"**/*.*"` on purpose, so it won't accidentally try to parse unrelated file types that might also be sitting in that Drive folder, while `show_progress=True` displays a progress bar as it works.

`raw_documents = loader.load()` actually reads the matched files from Drive and returns a list of `Document` objects, each holding the full text of one file plus its source path in `metadata`. This is the "Document Loading" stage of the RAG architecture, and it produces the raw material the rest of the pipeline works on.

`text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)` configures a splitter that tries to break text at natural boundaries paragraphs first, then sentences, then words, while keeping each chunk to roughly 500 characters. `chunk_size=500` keeps chunks small enough to be embedded precisely and retrieved individually, while `chunk_overlap=50` repeats the last 50 characters of one chunk at the start of the next, so a sentence split across a chunk boundary isn't completely lost in either half.

`documents = text_splitter.split_documents(raw_documents)` applies the splitter to every raw document loaded from Drive and returns the final list of small chunks. These chunks not the full original documents are what get embedded and stored in the vector database in the next cell.

Finally, the `print(...)` statement reports how many source files were found in Drive and how many chunks they produced. This is the key sanity check for this Drive-based setup specifically: if it prints `0` raw documents, the most likely cause is that `data_path` doesn't match where the files actually sit in Drive, and everything downstream needs to be re-checked before continuing.


In [8]:
# Cell 3: Google Drive Mount, Data Loading & Chunking
from google.colab import drive
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Mount Google Drive
drive.mount('/content/drive')

# Path to the dataset folder in Google Drive
data_path = "/content/drive/MyDrive/my_data"

# Load all .txt documents from the folder
loader = DirectoryLoader(
    data_path,
    glob="**/*.txt",
    show_progress=True
)

raw_documents = loader.load()

# Split documents into smaller semantic chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

documents = text_splitter.split_documents(raw_documents)

print(
    f"Loaded {len(raw_documents)} raw document(s) and "
    f"split into {len(documents)} chunks."
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


100%|██████████| 3/3 [00:00<00:00, 24.09it/s]

Loaded 3 raw document(s) and split into 36 chunks.


### Explanation — Cell 4: Embedding Model & Vector DB Indexing

`from langchain_huggingface import HuggingFaceEmbeddings` imports a wrapper class that runs a Hugging Face sentence-embedding model locally. Embeddings are what allow "semantic" search finding chunks that are conceptually related to a query, not just ones that share exact keywords. `from langchain_community.vectorstores import Chroma` imports the LangChain wrapper around the ChromaDB vector database, which is what actually stores the embeddings and performs the similarity search at query time.

`embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")` loads the `all-MiniLM-L6-v2` sentence-transformer model, which converts any piece of text into a 384-dimension numeric vector. This particular model is small, fast, free, and runs on CPU, which is why it's a common default for lab and prototype RAG systems rather than a large paid embedding API.

`vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings)` embeds every chunk produced in Cell 3 using the model above, then stores each resulting vector along with the original text and metadata inside a new in-memory Chroma collection. This is the indexing step of RAG: it converts our *Arcane* lore chunks into a searchable numeric index that later steps can query.

`retriever = vectorstore.as_retriever(search_kwargs={"k": 3})` wraps the vector store in a standard LangChain `Retriever` interface configured to return the top 3 most similar chunks for any given query. This `k=3` value is the "top-k retrieval value" mentioned in the report requirements, and it controls the trade-off between giving the LLM enough context to answer correctly and overwhelming it with irrelevant text.


In [9]:
# Cell 4: Embedding Model & Vector DB Indexing
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize open-source embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Store embeddings into Chroma vector database
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

# Set vectorstore as a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Explanation — Cell 5: Model Initialization and Domain System Prompt

`from langchain_groq import ChatGroq` imports the chat-model wrapper for Groq's hosted inference API this is the class that actually talks to the `llama-3.1-8b-instant` model over the network. `from langchain_core.prompts import ChatPromptTemplate` imports a class for building reusable, parameterized chat prompts, letting us define a prompt with placeholders like `{context}` and `{input}` once and have LangChain fill them in automatically for every query.

`llm = ChatGroq(model_name="openai/gpt-oss-20b", temperature=0)` creates a client for `gpt-oss-20b`, OpenAI's open-weight 20-billion-parameter model, hosted on Groq's fast inference infrastructure. Setting `temperature=0` makes the model's output deterministic and conservative it always picks the most likely next token which is what makes it stick closely to the retrieved context instead of improvising.

The `system_prompt = (...)` block is a multi-line string that defines the instructions the LLM receives on every call. It tells the model to act as a domain-specific assistant, to answer strictly from the provided context, and to explicitly refuse with a fixed fallback sentence if the answer isn't in the context. This system prompt is the main hallucination-mitigation mechanism in the whole system. The `"Context:\n{context}"` line at the end is a template placeholder that reserves a spot where the retrieved document chunks will be automatically inserted at query time without it, the LLM would receive no grounding text at all and would simply answer from its own pre-trained knowledge, defeating the purpose of RAG.

`prompt = ChatPromptTemplate.from_messages([("system", system_prompt), ("human", "{input}")])` builds a two-message chat template: a system message carrying the rules above, and a human message containing the placeholder `{input}` for the user's actual question. This mirrors how chat-based LLM APIs expect conversations to be formatted (system role plus user role), and it's what the chain in Cell 6 calls at runtime.


In [10]:
# Cell 5: Model Initialization and Domain System Prompt
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# Initialize the SLM
llm = ChatGroq(
    model_name="openai/gpt-oss-20b",
    temperature=0
)

# Custom domain system prompt
system_prompt = (
    "You are a specialized AI assistant for the user's uploaded domain.\n"
    "Answer questions strictly using ONLY the provided context below.\n"
    "If the answer cannot be found in the context, reply: 'I cannot answer based on the provided domain data.'\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

### Explanation — Cell 6: Pipeline Assembly

`from langchain_classic.chains import create_retrieval_chain` imports a factory function that wires a retriever and a "document-combining" chain together into one runnable pipeline this is the function that turns our separate pieces (retriever, prompt, LLM) into a single, callable RAG chain. `from langchain_classic.chains.combine_documents import create_stuff_documents_chain` imports a factory function that builds a chain which "stuffs" (concatenates) all retrieved documents into the prompt's `{context}` slot before calling the LLM. Stuffing is the simplest context-combination strategy, and it's appropriate here because our `k=3` chunks of at most 500 characters each comfortably fit inside the model's context window.

`combine_docs_chain = create_stuff_documents_chain(llm, prompt)` creates a chain that takes the retrieved documents plus the user's input, fills them into the `prompt` template from Cell 5, and sends the result to `llm`. This is the component that actually generates the final natural-language answer.

`rag_chain = create_retrieval_chain(retriever, combine_docs_chain)` wraps everything together: given a raw user question, it first calls `retriever` to fetch the top-3 relevant chunks, then passes them into `combine_docs_chain` to produce the answer. This single object, `rag_chain`, is the complete, end-to-end RAG system — from this point on, we only need to call `.invoke()` on it to ask questions.

Both imports come from `langchain_classic` rather than plain `langchain` because recent LangChain releases moved this pre-built retrieval-chain functionality into the separate `langchain-classic` package installed in Cell 1. If you see an `ImportError` here, it usually means either `langchain-classic` wasn't installed, or an old, cached version of `langchain` is shadowing it re-running Cell 1 and restarting the runtime resolves this.


In [11]:
# Cell 6: Pipeline Assembly
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Combine prompt and LLM to process context
combine_docs_chain = create_stuff_documents_chain(llm, prompt)

# Assemble full retrieval-augmented generation chain
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

### Explanation — Cell 7: Testing Your Custom Domain Chatbot (In-Domain Query)

`user_query = "Who is Jinx?"` defines an in-domain test question one whose answer genuinely exists in `arcane_character_profiles.txt`, since Jinx (Powder) is one of the profiled characters. This is Test Case 1, used to confirm the pipeline works correctly before moving on to the out-of-domain fallback and hallucination tests.

`response = rag_chain.invoke({"input": user_query})` runs the full RAG pipeline: it retrieves the top-3 relevant chunks for the query, inserts them into the prompt, and calls the LLM to generate an answer. `.invoke()` is LangChain's standard "run this chain once, synchronously" entry point, and the returned `response` dictionary contains both the generated answer and the source chunks used to produce it.

`print("--- DOMAIN QUERY ANSWER ---")` and `print(response["answer"])` display the LLM's generated answer, which lives under the `"answer"` key of the response dictionary.

`print("\n--- RETRIEVED SOURCE CHUNKS ---")` prints a header before listing the chunks that were retrieved, separating the generated answer from the supporting evidence in the console output for readability.

The loop `for i, doc in enumerate(response["context"]):` iterates over the list of `Document` objects under the `"context"` key — i.e., exactly the chunks the retriever fed to the LLM — which lets us audit which chunks the model actually used, essential for verifying an answer is truly grounded rather than hallucinated. Inside the loop, `print(f"Chunk {i+1} Source:", doc.metadata.get("source", "Unknown"))` prints the originating filename for each retrieved chunk, numbered starting at 1. This gives us traceability: for every claim in the answer, we can point back to the exact source file it came from, which is the core value proposition of a grounded RAG system over a plain LLM.


In [12]:
# Cell 7: Testing Your Custom Domain Chatbot
# Test Case 1: In-Domain Query
user_query = "Who is Jinx?"
response = rag_chain.invoke({"input": user_query})
print("--- DOMAIN QUERY ANSWER ---")
print(response["answer"])

print("\n--- RETRIEVED SOURCE CHUNKS ---")
for i, doc in enumerate(response["context"]):
    print(f"Chunk {i+1} Source:", doc.metadata.get("source", "Unknown"))

--- DOMAIN QUERY ANSWER ---
Jinx is the name given to Powder, Vi’s younger sister. After a childhood accident that killed several people, Vi abandoned her. Powder was then taken in by Silco, the crime boss of Zaun, who raised her as a weapon for his cause and renamed her Jinx. She became a brilliant but unstable inventor of chaotic weaponry and served as Silco’s enforcer and successor.

--- RETRIEVED SOURCE CHUNKS ---
Chunk 1 Source: /content/drive/MyDrive/my_data/arcane_character_profiles.txt
Chunk 2 Source: /content/drive/MyDrive/my_data/arcane_character_profiles.txt
Chunk 3 Source: /content/drive/MyDrive/my_data/arcane_character_profiles.txt


### Explanation — Optional Extra: Gradio Chat Interface

This cell is not part of the required lab template, but it wraps the finished `rag_chain` in a small web-based chat UI so the chatbot can be interacted with conversationally instead of one `print()` at a time.

`!pip install gradio -q` installs Gradio, a Python library for quickly building shareable web UIs, quietly so the install log doesn't clutter the notebook.

`import gradio as gr` imports the library under the conventional alias `gr`.

`def chatbot(message, history):` defines the function Gradio will call every time a user sends a message. It receives the new `message` plus the running `history` of the conversation (required by Gradio's `ChatInterface`, even though it isn't used here). Inside, `response = rag_chain.invoke({"input": message})` runs the exact same grounded RAG pipeline built above on whatever the user typed, and `return response["answer"]` sends just the generated answer text back to be displayed in the chat window.

`demo = gr.ChatInterface(fn=chatbot, title=..., description=...)` builds the chat widget itself, wiring the `chatbot` function in as the handler for every new message, and giving the interface a title and description so it's clear what it's for.

`demo.launch(share=True)` starts a local web server for the interface and, because `share=True`, also generates a temporary public link so the chatbot can be tested or shared without needing to expose your own machine or Colab session directly.


In [13]:
!pip install gradio -q

import gradio as gr

def chatbot(message, history):
    response = rag_chain.invoke({"input": message})
    return response["answer"]

demo = gr.ChatInterface(
    fn=chatbot,
    title="League of Legends: Arcane Lore\nRAG Chatbot",
    description="Ask any question about League of Legends Arcane Lore."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://90a954ad628e978096.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Task 2: Out-of-Domain Fallback Test

`ood_query = "..."` defines a question that has nothing to do with our *Arcane* lore dataset it asks about real-world space exploration instead of Piltover, Zaun, or Hextech. This directly tests whether the `temperature=0` plus explicit-refusal system prompt from Cell 5 successfully prevents the model from making up an answer when no relevant context exists.

`ood_response = rag_chain.invoke({"input": ood_query})` runs the exact same grounded pipeline (the baseline configuration) against the out-of-domain question. Because `retriever` will still return its top-3 "closest" chunks even when none of them are actually relevant, this checks whether the LLM correctly recognizes that those chunks don't answer the question and falls back rather than fabricating a connection.

`print(ood_response["answer"])` displays the model's response to the out-of-domain query. We expect this to print the required fallback message: *"I cannot answer based on the provided domain data."*


In [14]:
# Task 2: Out-of-Domain Fallback Test (baseline configuration: temperature=0, with fallback rule)
ood_query = "What year did NASA's Apollo 11 mission land on the moon?"
ood_response = rag_chain.invoke({"input": ood_query})
print("--- OUT-OF-DOMAIN QUERY ---")
print(ood_query)
print("\n--- MODEL RESPONSE (baseline) ---")
print(ood_response["answer"])

--- OUT-OF-DOMAIN QUERY ---
What year did NASA's Apollo 11 mission land on the moon?

--- MODEL RESPONSE (baseline) ---
I cannot answer based on the provided domain data.


# **Observe & Compare**

This is the section after I re-running Cell 5 with the modified temperature and system_prompt, and re-invoking rag_chain with the same out-of-domain query.

**Baseline (temperature=0, with fallback rule) — Task 2 output:**

*"I cannot answer based on the provided domain data."*

**Stress-test (temperature=1.0, fallback rule removed) — Task 3 output:**

*"I'm sorry, I don't have that information."*

# **Comparison of Responses Under Both Configurations**

Under the baseline configuration (temperature=0, fallback rule included), I asked the out-of-domain question about Apollo 11, and the chatbot replied with the exact scripted fallback message: "I cannot answer based on the provided domain data." This was expected, since the prompt directly told it what to say when the answer isn't in the retrieved context.

Under the stress-test configuration (temperature=1.0, fallback rule removed), I asked the exact same question again. The chatbot still did not answer the question directly it replied, "I'm sorry, I don't have that information," instead of making up a year or any details about the mission.

Comparison of the two shows that the model has maintained its reliability in either scenario. The model did not lose its reliability, and it did not make use of information which could not be validated. In fact, the only difference between the two responses is the refusal statement. Without having the fallback sentence ready, the model needed to make one of its own, but it still knew that it did not have an answer to the question in the retrieved context.

An increase in the temperature level similarly did not lead to any instances of the model hallucinating in this instance. The key impact here is that the higher temperature level increased the unpredictability of the wording used, because with a higher temperature, the model tends to use improbable words instead of the most likely one. However, the increase in the temperature level did not make the model deviate from the instruction of using the given context.
